# **01 Populando a Base de Dados**

Este arquivo utiliza a biblioteca faker para criar dados ficticios para a base de dados monitoramento_db

---

### **Escopo do Notebook:**

- Conectando ao Banco de Dados
- Importando bibliotecas
- Populando tabela cliente
- Populando tabela veiculos
- Populando tabela centros_distribuicao
- Populando tabela fato_envio
- Populando tabela fato_status_entrega
- Retornando head do banco em um Data Frame

---

# **Código:**

### **Conectando ao Banco de Dados**

In [1]:
# Conectando ao banco de dados

import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=monitoramento_db;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()

### **Importando bibliotecas**

In [2]:
# Importando bibliotecas restantes

from faker import Faker
import random
from datetime import timedelta

# Dizendo a biblioteca para que use apenas dados brasileiros
fake = Faker("pt_BR")

### **Populando tabela cliente**

In [3]:
# Insert na tabela cliente

for _ in range(100):

    nome = fake.name()
    cidade = fake.city()
    estado = fake.estado_sigla()

    cursor.execute("""
        INSERT INTO clientes
        (nome,cidade,estado)
        VALUES (?,?,?)
    """, nome, cidade, estado)

conn.commit()

### **Populando tabela veiculos**

In [ ]:
veiculos = [
    ("Fiat Mobi", 1),
    ("Fiat Uno", 1),
    ("Volkswagen Gol", 1),
    ("Volkswagen Polo", 1),
    ("Chevrolet Onix", 1),
    ("Chevrolet Prisma", 1),
    ("Hyundai HB20", 1),
    ("Toyota Corolla", 1),
    ("Honda Civic", 1),
    ("Jeep Renegade", 1),
    ("Nissan Versa", 1),
    ("Renault Kwid", 1),
    ("Ford Ka", 1),
    ("Peugeot 208", 1),
    ("Citroen C3", 1),

    ("Fiat Toro", 2),
    ("Toyota Hilux", 2),
    ("Chevrolet S10", 2),
    ("Ford Ranger", 2),
    ("Volkswagen Amarok", 2),

    ("Mercedes-Benz Sprinter", 3),
    ("Renault Master", 3),
    ("Fiat Ducato", 3),
    ("Volkswagen Delivery 11.180", 3),
    ("Mercedes-Benz Atego 1719", 3),
    ("Volkswagen Constellation 24.280", 3),
    ("Scania P320", 3),
    ("Volvo VM 270", 3),
    ("Iveco Tector 240E30", 3),
    ("DAF XF 480", 3)
]

for modelo, categoria in veiculos:

    placa = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=3))
    placa += ''.join(random.choices('0123456789', k=4))

    if categoria == 1:
        capacidade = 100
    elif categoria == 2:
        capacidade = 1000
    else: 
        capacidade = 2000

    cursor.execute("""
        INSERT INTO veiculos
        (modelo, placa, capacidade_kg, categoria)
        VALUES (?, ?, ?, ?)
    """, modelo, placa, capacidade, categoria)

conn.commit()

### **Populando tabela centros_distribuicao**

In [5]:
# Insert na tabela centros_distribuicao

cds = [
    ('CD São Paulo','São Paulo','SP'),
    ('CD Campinas','Campinas','SP'),
    ('CD Rio','Rio de Janeiro','RJ'),
    ('CD Curitiba','Curitiba','PR'),
    ('CD Salvador','Salvador','BA'),
    ('CD Recife','Recife','PE'),
    ('CD Fortaleza','Fortaleza','CE'),
    ('CD Goiânia','Goiânia','GO'),
    ('CD Manaus','Manaus','AM'),
    ('CD Porto Alegre','Porto Alegre','RS')
]

for cd in cds:

    cursor.execute("""
        INSERT INTO centros_distribuicao
        (nome_hub,cidade,estado)
        VALUES (?,?,?)
    """, cd)

conn.commit()

### **Populando tabela fato_envios**

In [6]:
# Insert na tabela fato envios

for _ in range(1000):

    cliente = random.randint(1,100)

    # Seleciona uma data nos últimos seis meses (-6M) 
    postagem = fake.date_time_between(
        start_date='-6M',
        end_date='now'
    )

    # Postagem + valor aleatório de 1 a 15
    previsao = postagem + timedelta(
        days=random.randint(1,15)
    )

    frete = round(random.uniform(25,1500),2)

    peso = round(random.uniform(0.5,2000),3)

    if peso <= 100:
        veiculo = random.randint(1,15)
    elif peso <= 1000:
        veiculo = random.randint(16,20)
    else:
        veiculo = random.randint(21,30)

    cursor.execute("""
        INSERT INTO fato_envios
        (
            id_cliente,
            id_veiculo,
            data_postagem,
            data_previsao_entrega,
            valor_frete,
            peso_carga
        )
        VALUES (?,?,?,?,?,?)
    """,
        cliente,
        veiculo,
        postagem,
        previsao,
        frete,
        peso
    )

conn.commit()

### **Populando tabela fato_status_entrega**

In [7]:
status_fluxo = [
    'Postado',
    'Recebido no CD',
    'Em transferência',
    'Saiu para entrega',
    'Entregue'
]

for envio in range(1,1001):                # 1, 1001 pois foram gerados 1000 inserts na tabela fatos envio

    data_base = fake.date_time_between(
        start_date='-6M',
        end_date='now'
    )

    quantidade_status = random.randint(3,5)

    for i in range(quantidade_status):

        cursor.execute("""
            INSERT INTO fato_status_entrega
            (
                id_envio,
                id_cd,
                status,
                ultima_atualizacao
            )
            VALUES (?,?,?,?)
        """,
            envio,
            random.randint(1,10),
            status_fluxo[i],                  # Percorre em sequência cada valor da lista fixa
            data_base + timedelta(hours=i*12) # Cada status só pode mudar 15 horas após o anterior
        )

conn.commit()

### **Retornando head do banco em um Data Frame**

In [8]:
import pandas as pd

pd.read_sql(
    "SELECT TOP 10 * FROM fato_status_entrega",
    conn
)

C:\Users\Igor Cruz\AppData\Local\Temp\ipykernel_32400\392947252.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,id_status,id_envio,id_cd,status,ultima_atualizacao
0,1,1,4,Postado,2026-02-15 15:13:36
1,2,1,9,Recebido no CD,2026-02-16 03:13:36
2,3,1,2,Em transferência,2026-02-16 15:13:36
3,4,1,10,Saiu para entrega,2026-02-17 03:13:36
4,5,2,4,Postado,2026-06-03 09:50:37
5,6,2,1,Recebido no CD,2026-06-03 21:50:37
6,7,2,6,Em transferência,2026-06-04 09:50:37
7,8,2,9,Saiu para entrega,2026-06-04 21:50:37
8,9,2,3,Entregue,2026-06-05 09:50:37
9,10,3,8,Postado,2026-02-08 01:22:43
